### Jupyter notebook to get the expanded organ masks, whole body masks, head masks, postprocessed tissue masks for quantification
#### What do you need:
##### 1. Downsampled raw image, saved as nii.gz file
##### 2. Organ mask from the Tissue Module for the downsampled raw image, saved as nii.gz file
##### 3. Tissue mask from the Tissue Module which is downsampled to the same resolution as the above organ mask

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import cv2
import scipy
import nibabel as nib
from skimage.segmentation import expand_labels

In [ ]:
# For overlay visualization
from ipywidgets import interact, IntSlider
def show_slice(raw, mask, slice_idx):
    plt.figure(figsize=(12, 6))
    plt.imshow(raw[:, :, slice_idx], cmap='gray', vmin=0, vmax=2000) #, vmin=0, vmax=1
    plt.imshow(mask[:, :, slice_idx], cmap="Reds",alpha=0.4)  # overlay mask with transparency
    plt.axis('off')
    plt.title(f"Slice {slice_idx}")
    plt.show()

In [ ]:
organ_mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/organmask_xy40z10.nii.gz"
raw_image_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/C00_xy40z10.nii.gz"
tissue_mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/tissuemask_xy40z10.nii.gz"

In [ ]:
## check the sizes of different masks and raw image to be same
raw_image= nib.load(raw_image_path).get_fdata()
organ_mask = nib.load(organ_mask_path).get_fdata()
tissue_mask = nib.load(tissue_mask_path).get_fdata()

In [ ]:
## define the path for saving expanded organ mask, whole body mask and postprocessed tissue mask
organ_mask_grow_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/organmask_grow_xy40z10.nii.gz"
wb_mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/wholebodymask_xy40z10.nii.gz"

### Grow organ mask for quantification

In [ ]:
def grow_organ_mask(np_organ_mask, distance=8):

    # grow the mask for organs without brain
    np_organ_mask[np_organ_mask==5]=0
    np_organ_mask = expand_labels(np_organ_mask, distance=distance) 
    return np_organ_mask
organ_mask_grow = grow_organ_mask(organ_mask)

In [ ]:
# Visualize the organ mask by overlaying it on the raw image
interact(
    lambda slice_idx: show_slice(raw_image, organ_mask_grow, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=209), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [ ]:
affmat = np.eye(4)
affmat[0,0] = affmat[1,1] = -1
NiftiObject = nib.Nifti1Image(organ_mask_grow, affine=affmat)
nib.save(NiftiObject, organ_mask_grow_path)

### Create wholebody mask

In [ ]:
init_wbmask  = np.zeros_like(raw_image)
init_wbmask[raw_image>160]=1 # By thresholding first to seperate background and foreground


In [ ]:
interact(
    lambda slice_idx: show_slice(raw_image, init_wbmask, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=250), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [ ]:
def get_wb_mask(arr_thresh, arr_organmask):
    arr_wb = arr_thresh + arr_organmask
    arr_wb[arr_wb>0]=1
    return arr_wb

wbmask = get_wb_mask(init_wbmask, organ_mask)

In [ ]:
interact(
    lambda slice_idx: show_slice(raw_image, wbmask, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

In [ ]:
def dilate_vol(vol, dilate_iter):
    vol_dilate = scipy.ndimage.binary_dilation(vol, iterations = dilate_iter)
    return vol_dilate

def erode_vol(vol, erode_iter):
    vol_erode = scipy.ndimage.binary_erosion(vol, iterations = erode_iter)
    return vol_erode

wbmask_pro = dilate_vol(erode_vol(wbmask, 2), 8)  # with dilate and erode operation, remove noise outside mouse bady and ensure the mask covers whole body

In [ ]:
interact(
    lambda slice_idx: show_slice(raw_image, wbmask_pro, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=250), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [ ]:
affmat = np.eye(4)
affmat[0,0] = affmat[1,1] = -1
NiftiObject = nib.Nifti1Image(wbmask_pro.astype(np.uint8), affine=affmat)
nib.save(NiftiObject,wb_mask_path)

## Postprocessing tissue mask with wholebody mask and organ mask

In [ ]:
raw_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/C00_xy40z10.nii.gz"
mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/tissuemask_xy40z10.nii.gz"

raw_img = nib.load(raw_path).get_fdata()
mask_img = nib.load(mask_path).get_fdata()
raw_img.shape, raw_img.min(), raw_img.max(), mask_img.shape, np.unique(mask_img)

((465, 1110, 204),
 np.float64(24.0),
 np.float64(28871.0),
 (465, 1110, 204),
 array([0., 1., 2., 3., 4.]))

In [107]:
interact(
    lambda slice_idx: show_slice(raw_img, mask_img, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_img.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=203), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [ ]:
wbmask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/wholebodymask_xy40z10.nii.gz"
organmask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/organmask_xy40z10.nii.gz"

wbmask_img = nib.load(wbmask_path).get_fdata()
organmask_img = nib.load(organmask_path).get_fdata()
wbmask_img.shape, np.unique(wbmask_img), organmask_img.shape, np.unique(organmask_img)

((465, 1110, 204),
 array([0., 1.]),
 (465, 1110, 204),
 array([ 0.,  1.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
        14., 15., 16., 19., 20., 21., 22., 23., 24., 25., 26., 27.]))

In [109]:
mask_img = mask_img * wbmask_img

In [110]:
interact(
    lambda slice_idx: show_slice(raw_img, mask_img, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_img.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=203), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [111]:
mask_img = mask_img * (organmask_img==0)

In [112]:
interact(
    lambda slice_idx: show_slice(raw_img, mask_img, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_img.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=203), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [113]:
print(np.unique(mask_img))
affmat = np.eye(4)
affmat[0,0] = affmat[1,1] = -1
NiftiObject = nib.Nifti1Image(mask_img.astype(np.uint8), affine=affmat)
nib.save(NiftiObject,r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250820_PGP_9-5_HFD_4x_5566_ventral_9by24_80-100-100_6um_Mstitched_lower_part_faulty/tissuemask_xy40z10.nii.gz")

[0. 1. 2. 3. 4.]


### Create head mask

In [30]:
def zslices_to_nifti(path_source):

    z_slices_sorted = sorted(os.listdir(path_source))
    image = cv2.imread(path_source + z_slices_sorted[0], 2) #load one tiff
    bb_y, bb_x = image.shape
    bb_z = len(z_slices_sorted)
    canvas = np.zeros((bb_y,bb_x,bb_z),np.uint16)
    for z,z_slice_name in enumerate(z_slices_sorted):
        #z_slice_name = z_slice_name.replace('C00',channelname) # file names were only saved for C00
        image = cv2.imread(path_source + z_slice_name, 2)
        canvas[:,:,z] = image
    return canvas


In [27]:
organ_mask_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/organ_pred.nii.gz"
organ_mask = nib.load(organ_mask_path).get_fdata()
#raw_slice_path = r"/ictstr01/groups/iterm/Ying/HFD/data/Downsampled/UCHL1_chow_977_1x_fused_whole_Arivis_Export/C01/xy10z10/"
#raw_image = zslices_to_nifti(raw_slice_path)
raw_image_path = r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/C00_xy40z10.nii.gz"
raw_image= nib.load(raw_image_path).get_fdata()

raw_image.shape, organ_mask.shape

((511, 1110, 210), (511, 1110, 210))

In [28]:
raw_thresh = np.zeros_like(raw_image)
raw_thresh[raw_image>180]=1
raw_thresh.shape

(511, 1110, 210)

In [29]:
interact(
    lambda slice_idx: show_slice(raw_image, raw_thresh, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=209), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [30]:
brain_mask = np.zeros_like(organ_mask)
brain_mask[organ_mask==5]=1
print(np.unique(brain_mask))
structure = np.ones((3, 3, 6), dtype=int)
brain_mask = scipy.ndimage.binary_dilation(brain_mask, structure, iterations=50).astype(np.uint8)

[0. 1.]


In [31]:
brain_mask.shape

(511, 1110, 210)

In [32]:
merge_head_mask = raw_thresh * brain_mask
merge_head_mask.shape

(511, 1110, 210)

In [33]:
interact(
    lambda slice_idx: show_slice(raw_image, merge_head_mask, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=209), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [34]:
merge_head_mask_fill = scipy.ndimage.binary_dilation(scipy.ndimage.binary_fill_holes(merge_head_mask, axes = 2), iterations=5).astype(np.uint8)

In [35]:
interact(
    lambda slice_idx: show_slice(raw_image, merge_head_mask_fill, slice_idx),
    slice_idx=IntSlider(min=0, max=raw_image.shape[2]-1, step=1, value=0)
)

interactive(children=(IntSlider(value=0, description='slice_idx', max=209), Output()), _dom_classes=('widget-i…

<function __main__.<lambda>(slice_idx)>

In [36]:
affmat = np.eye(4)
affmat[0,0] = affmat[1,1] = -1
NiftiObject = nib.Nifti1Image(merge_head_mask_fill, affine=affmat)
nib.save(NiftiObject,r"/ictstr01/groups/iterm/Ying/HFD/data/4x_scans/250515_Uchl1_HFD_961_4x_ventral_11by24_6um_80_100_100_MStitched/headmask_xy40z10.nii.gz")